In [7]:
import os, sys, json
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()


## 1. JSON con System Prompt

In [2]:
if "client" not in dir():
    import sys, json; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()

import json

def extract_json(text: str) -> dict:
    """Extrae y parsea JSON de la respuesta de Claude."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

system_json = """
Responde UNICAMENTE con JSON valido, sin texto adicional ni bloques de codigo.
"""

user_msg = """
Extrae la informacion del siguiente texto en JSON con campos: 
nombre, edad, profesion, ciudad.

Texto: "Maria Garcia tiene 34 años, trabaja como ingeniera de software 
y vive en Barcelona desde hace 5 años."
"""

resp = client.messages.create(
    model=DEFAULT_MODEL, max_tokens=300,
    system=system_json,
    messages=[{"role": "user", "content": user_msg}]
)

raw = resp.content[0].text
print("Respuesta raw:", raw)

data = extract_json(raw)
print("\nJSON parseado:")
print(json.dumps(data, ensure_ascii=False, indent=2))


Respuesta raw: ```json
{
  "nombre": "Maria Garcia",
  "edad": 34,
  "profesion": "ingeniera de software",
  "ciudad": "Barcelona"
}
```

JSON parseado:
{
  "nombre": "Maria Garcia",
  "edad": 34,
  "profesion": "ingeniera de software",
  "ciudad": "Barcelona"
}


## 2. Prefill para Forzar JSON

In [3]:
# El prefill inicia la respuesta del asistente con '{'
# Esto garantiza que Claude continúe con JSON

resp = client.messages.create(
    model=DEFAULT_MODEL, max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": """Genera una lista de 3 productos tecnológicos con sus características.
            Formato JSON: {"productos": [{"nombre": ..., "precio": ..., "categoria": ...}]}"""
        },
        {
            "role": "assistant",
            "content": "{"  # Prefill: iniciamos la respuesta del asistente
        }
    ]
)

# La respuesta no incluye el '{' inicial, lo añadimos
json_str = "{" + resp.content[0].text
print("JSON completo:")
data = json.loads(json_str)
print(json.dumps(data, ensure_ascii=False, indent=2))


JSON completo:
{
  "productos": [
    {
      "nombre": "Smartphone Samsung Galaxy S24",
      "precio": 899.99,
      "categoria": "Telefonía",
      "características": {
        "pantalla": "6.2 pulgadas AMOLED",
        "procesador": "Snapdragon 8 Gen 3",
        "ram": "8GB",
        "almacenamiento": "256GB",
        "camara": "50MP principal"
      }
    },
    {
      "nombre": "Laptop Dell XPS 15",
      "precio": 1499.99,
      "categoria": "Computación",
      "características": {
        "pantalla": "15.6 pulgadas 4K",
        "procesador": "Intel Core i7-13700H",
        "ram": "16GB DDR5",
        "almacenamiento": "512GB SSD",
        "tarjeta_grafica": "NVIDIA RTX 4050"
      }
    },
    {
      "nombre": "Apple AirPods Pro 2",
      "precio": 249.99,
      "categoria": "Audio",
      "características": {
        "cancelacion_ruido": "Activa",
        "autonomia": "6 horas",
        "conectividad": "Bluetooth 5.3",
        "resistencia": "IPX4",
        "chip": "H2"
   

## 3. Ejercicio: Pipeline de Extracción de Datos

In [8]:
reviews = [
    "El iPhone 15 es increible, la camara es perfecta. Lo compre por $999. 5 estrellas",
    "Decepcionante. La Samsung Galaxy no duro ni un año. Pague $750. Solo 2 estrellas.",
    "El MacBook Pro vale cada centavo. Potente, elegante. $1299. Me encanta. 5/5",
]

system_extractor = """
Eres un extractor de datos. Para cada resena, devuelve JSON con:
{
  "producto": string,
  "sentimiento": "positivo" | "negativo" | "neutral",
  "precio": number,
  "puntuacion": number  (en escala 1-5)
}
Solo JSON, sin texto adicional.
"""

resultados = []
for review in reviews:
    resp = client.messages.create(
        model=DEFAULT_MODEL, max_tokens=200,
        system=system_extractor,
        messages=[{"role": "user", "content": review}]
    )
    try:
        data = extract_json(resp.content[0].text)
        resultados.append(data)
        print(f"OK: {data}")
    except (json.JSONDecodeError, IndexError) as e:
        print(f"Error parseando: {resp.content[0].text[:100]}")

print("\nResumen:")
print(f"  Positivos: {sum(1 for r in resultados if r.get('sentimiento') == 'positivo')}")
print(f"  Negativos: {sum(1 for r in resultados if r.get('sentimiento') == 'negativo')}")
precios = [r.get('precio', 0) for r in resultados if r.get('precio')]
if precios:
    print(f"  Precio promedio: ${sum(precios)/len(precios):.0f}")
else:
    print("  Sin datos de precio")


OK: {'producto': 'iPhone 15', 'sentimiento': 'positivo', 'precio': 999, 'puntuacion': 5}
OK: {'producto': 'Samsung Galaxy', 'sentimiento': 'negativo', 'precio': 750, 'puntuacion': 2}
OK: {'producto': 'Samsung Galaxy', 'sentimiento': 'negativo', 'precio': 750, 'puntuacion': 2}
OK: {'producto': 'MacBook Pro', 'sentimiento': 'positivo', 'precio': 1299, 'puntuacion': 5}

Resumen:
  Positivos: 2
  Negativos: 1
  Precio promedio: $1016
OK: {'producto': 'MacBook Pro', 'sentimiento': 'positivo', 'precio': 1299, 'puntuacion': 5}

Resumen:
  Positivos: 2
  Negativos: 1
  Precio promedio: $1016
